# Giảm chiều dữ liệu MNIST – So sánh Baseline vs Sau giảm chiều

**Đề tài:**
- Thực hiện giảm chiều (PCA) cho bộ MNIST.
- Đánh giá **baseline** (dữ liệu gốc) và **sau giảm chiều**.
- So sánh: **tiêu thụ bộ nhớ**, **thời gian xử lý** (fit + predict), **hao hụt độ chính xác**.

## 1. Import và tải dữ liệu

In [ ]:
import sys
sys.path.insert(0, '.')

from lib import (
    MNISTDataLoader,
    MNISTClassifier,
    DimensionalityReducer,
    measure_array_memory_mb,
    run_and_measure_seconds,
    plot_comparison_reduction,
    print_comparison_table,
    plot_confusion_matrix,
    print_classification_report,
)
import matplotlib.pyplot as plt

In [ ]:
loader = MNISTDataLoader(normalize=True, flatten=True)
X_train, y_train, X_test, y_test = loader.load()

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Số chiều gốc: {X_train.shape[1]}")

## 2. Baseline – Đánh giá trên dữ liệu gốc (784 chiều)

In [ ]:
baseline_memory_mb = measure_array_memory_mb(X_train)
print(f"Bộ nhớ X_train (baseline): {baseline_memory_mb:.4f} MB")

In [ ]:
clf_baseline = MNISTClassifier(model_type="logistic", random_state=42)

_, time_fit_baseline = run_and_measure_seconds(lambda: clf_baseline.fit(X_train, y_train))
_, time_predict_baseline = run_and_measure_seconds(lambda: clf_baseline.predict(X_test))

acc_baseline = clf_baseline.score(X_test, y_test)

baseline_metrics = {
    "memory_mb": baseline_memory_mb,
    "time_fit_s": time_fit_baseline,
    "time_predict_s": time_predict_baseline,
    "accuracy": acc_baseline,
}
print(f"Baseline - Fit: {time_fit_baseline:.4f} s, Predict: {time_predict_baseline:.4f} s, Accuracy: {acc_baseline:.4f}")

## 3. Giảm chiều bằng PCA

In [ ]:
n_components = 0.95
reducer = DimensionalityReducer(n_components=n_components, random_state=42)
X_train_reduced = reducer.fit_transform(X_train)
X_test_reduced = reducer.transform(X_test)

print(f"Số chiều sau giảm: {reducer.n_components_}")
print(f"Tổng phương sai giữ lại: {reducer.total_explained_variance_ratio():.4f}")

In [ ]:
reduced_memory_mb = measure_array_memory_mb(X_train_reduced)
print(f"Bộ nhớ X_train (sau giảm chiều): {reduced_memory_mb:.4f} MB")

## 4. Đánh giá trên dữ liệu sau giảm chiều

In [ ]:
clf_reduced = MNISTClassifier(model_type="logistic", random_state=42)

_, time_fit_reduced = run_and_measure_seconds(lambda: clf_reduced.fit(X_train_reduced, y_train))
_, time_predict_reduced = run_and_measure_seconds(lambda: clf_reduced.predict(X_test_reduced))

acc_reduced = clf_reduced.score(X_test_reduced, y_test)

reduced_metrics = {
    "memory_mb": reduced_memory_mb,
    "time_fit_s": time_fit_reduced,
    "time_predict_s": time_predict_reduced,
    "accuracy": acc_reduced,
}
print(f"Sau giảm chiều - Fit: {time_fit_reduced:.4f} s, Predict: {time_predict_reduced:.4f} s, Accuracy: {acc_reduced:.4f}")

## 5. So sánh: Bộ nhớ, Thời gian, Độ chính xác

In [ ]:
print_comparison_table(baseline_metrics, reduced_metrics, reduced_name="Sau giảm chiều (PCA)")

In [ ]:
fig = plot_comparison_reduction(
    baseline_metrics,
    reduced_metrics,
    title="So sánh Baseline (784 chiều) vs Sau giảm chiều PCA",
)
plt.show()

## 6. Báo cáo phân lớp và Confusion Matrix (sau giảm chiều)

In [ ]:
y_pred_reduced = clf_reduced.predict(X_test_reduced)
class_names = loader.get_class_names()

print("Báo cáo phân lớp (mô hình trên dữ liệu sau giảm chiều):")
print_classification_report(y_test, y_pred_reduced, class_names=class_names)

In [ ]:
fig = plot_confusion_matrix(y_test, y_pred_reduced, class_names=class_names, title="Confusion Matrix – Sau giảm chiều PCA")
plt.show()

## 7. (Tùy chọn) Thử nhiều mức giảm chiều

So sánh hao hụt độ chính xác và tiết kiệm bộ nhớ khi thay đổi số chiều PCA.

In [ ]:
results = []
for n_comp in [0.99, 0.95, 0.90, 0.80, 50, 30]:
    r = DimensionalityReducer(n_components=n_comp, random_state=42)
    X_tr = r.fit_transform(X_train)
    X_te = r.transform(X_test)
    mem = measure_array_memory_mb(X_tr)
    clf = MNISTClassifier(model_type="logistic", random_state=42)
    clf.fit(X_tr, y_train)
    acc = clf.score(X_te, y_test)
    results.append({"n_components": n_comp, "dims": r.n_components_, "memory_mb": mem, "accuracy": acc})

print("n_components (số hoặc % var)	Số chiều	Bộ nhớ (MB)	Accuracy")
for row in results:
    print(f"{row['n_components']}				{row['dims']}		{row['memory_mb']:.4f}		{row['accuracy']:.4f}")